<a href="https://colab.research.google.com/github/amirhamrouni/Alala/blob/main/docs/MoneyPrinterTurbo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Run MoneyPrinterTurbo on Google Colab

This notebook installs [MoneyPrinterTurbo](https://github.com/harry0703/MoneyPrinterTurbo) in an isolated Python 3.11 environment and launches its WebUI. Colab runtimes are temporary, so generated files are removed when the runtime is reset.

## 1. Install MoneyPrinterTurbo

The setup is safe to run again: it clones the repository on the first run and updates it on later runs. Dependencies are installed in the project's virtual environment to avoid conflicts with Colab's preinstalled packages.

In [2]:
import os, subprocess, time, urllib.request, re
from pathlib import Path
REPO_DIR = Path('/content/MoneyPrinterTurbo')
REPO_URL = 'https://github.com/harry0703/MoneyPrinterTurbo.git'
NGROK_TOKEN = '3K04LwvnusYFnv78jj00awjG2Gn_7fE2BfvpxKrU3eVgWCDx1'
PORT = 8501

def say(x): print(x, flush=True)
def run(cmd, **kw):
    say('$ ' + ' '.join(map(str, cmd)))
    return subprocess.run(cmd, check=True, **kw)

say('1/4 prepare')
if not (REPO_DIR/'.git').is_dir():
    if REPO_DIR.exists(): run(['rm','-rf',str(REPO_DIR)])
    run(['git','clone','--depth','1',REPO_URL,str(REPO_DIR)])
os.chdir(REPO_DIR)
run(['python','-m','pip','install','-q','uv','pyngrok'])
subprocess.run(['uv','python','install','3.11'], check=False)
run(['uv','sync','--frozen','--python','3.11'])

say('2/4 start WebUI')
subprocess.run(['bash','-lc', f'fuser -k {PORT}/tcp >/dev/null 2>&1 || true'], check=False)
time.sleep(1)
log = open('/content/moneyprinterturbo-webui.log','w',encoding='utf-8')
proc = subprocess.Popen(['uv','run','streamlit','run','webui/Main.py',f'--server.port={PORT}','--server.address=0.0.0.0','--browser.gatherUsageStats=False','--client.toolbarMode=minimal','--server.showEmailPrompt=False'], cwd=REPO_DIR, stdout=log, stderr=subprocess.STDOUT, text=True)
end=time.time()+120
while time.time()<end:
    if proc.poll() is not None: break
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/_stcore/health', timeout=3) as r:
            if r.status==200:
                say('WebUI health 200')
                break
    except Exception: pass
    time.sleep(2)
else:
    log.flush(); say(Path('/content/moneyprinterturbo-webui.log').read_text(errors='replace')[-4000:]); raise RuntimeError('WebUI failed')

say('3/4 open ngrok')
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
try: ngrok.kill()
except Exception: pass
tunnel = ngrok.connect(addr=f'http://127.0.0.1:{PORT}', proto='http', bind_tls=True)
say('URL=' + tunnel.public_url)
say('4/4 ready. لا توقف هذه الخلية')
while True:
    time.sleep(60)


1/4 prepare
$ python -m pip install -q uv pyngrok
$ uv sync --frozen --python 3.11
2/4 start WebUI
WebUI health 200
3/4 open ngrok
URL=https://rule-unnamed-could.ngrok-free.dev
4/4 ready. لا توقف هذه الخلية


KeyboardInterrupt: 

## 2. Configure the Colab tunnel

MoneyPrinterTurbo itself does not require ngrok for normal local use. This notebook uses ngrok only because the Streamlit server runs inside a remote Colab runtime.

Create an account and copy your authentication token from the [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken). The next cell reads it without displaying or storing it in the notebook.

In [ ]:
from getpass import getpass

from pyngrok import ngrok

# Close tunnels from earlier runs before configuring a new one.
ngrok.kill()

ngrok_token = getpass("Enter your ngrok authentication token: ").strip()
if not ngrok_token:
    raise ValueError("An ngrok authentication token is required")
ngrok.set_auth_token(ngrok_token)
del ngrok_token
print("ngrok authentication configured")

## 3. Launch the WebUI

The cell waits for Streamlit to become healthy before creating the public URL. If startup fails, it includes the recent server log in the error. After opening the URL, use **Settings** to configure the required model and media API keys. To use OpenRouter free LLMs, select **OpenRouter**, enter your OpenRouter API key, and keep the default `minimax/minimax-m3:free` model or choose another model from https://openrouter.ai/models?variant=free.

In [ ]:
import time
from pathlib import Path
import subprocess
from urllib.error import URLError
from urllib.request import urlopen

from pyngrok import ngrok

# Sanity checks before starting
REPO_DIR = Path("/content/MoneyPrinterTurbo")
if not REPO_DIR.is_dir():
    raise RuntimeError(f"Repo directory does not exist: {REPO_DIR}")
if not (REPO_DIR / "webui" / "Main.py").is_file():
    raise RuntimeError(f"webui/Main.py not found in {REPO_DIR}")

PORT = 8501
LOG_PATH = Path("/content/moneyprinterturbo-webui.log")

# Make this cell safe to rerun by closing the previous process, log handle, and tunnel.
previous_streamlit_proc = globals().get("streamlit_proc")
if previous_streamlit_proc is not None and previous_streamlit_proc.poll() is None:
    previous_streamlit_proc.terminate()
    previous_streamlit_proc.wait(timeout=10)
previous_streamlit_log = globals().get("streamlit_log")
if previous_streamlit_log is not None and not previous_streamlit_log.closed:
    previous_streamlit_log.close()

try:
    ngrok.kill()
except Exception:
    pass

streamlit_log = LOG_PATH.open("w", encoding="utf-8")
streamlit_proc = subprocess.Popen(
    [
        "uv",
        "run",
        "streamlit",
        "run",
        "webui/Main.py",
        f"--server.port={PORT}",
        "--server.address=0.0.0.0",
        "--browser.gatherUsageStats=False",
        "--client.toolbarMode=minimal",
        "--server.showEmailPrompt=False",
    ],
    cwd=REPO_DIR,
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
    text=True,
)

# Poll the health endpoint because startup time varies across Colab runtimes.
deadline = time.time() + 90
server_ready = False
while time.time() < deadline:
    if streamlit_proc.poll() is not None:
        break
    try:
        with urlopen(f"http://127.0.0.1:{PORT}/_stcore/health", timeout=2) as response:
            server_ready = response.status == 200
    except (URLError, TimeoutError):
        pass
    if server_ready:
        break
    time.sleep(2)

if not server_ready:
    streamlit_log.flush()
    recent_log = LOG_PATH.read_text(encoding="utf-8", errors="replace")[-4000:]
    raise RuntimeError(f"Streamlit failed to start. Recent log:\n{recent_log}")

# Use an explicit IPv4 URL because pyngrok may otherwise route localhost through ::1.
public_tunnel = ngrok.connect(addr=f"http://127.0.0.1:{PORT}", proto="http", bind_tls=True)
print("MoneyPrinterTurbo is ready:")
print(public_tunnel.public_url)
print(f"Server log: {LOG_PATH}")